# 05. Итоговые таблицы, сравнение KAN с контролем, графики
Ноутбук ничего не обучает: он читает `results/*.json` и `results/ranks/*.npy`, результаты пишет в `results/summary/` (CSV, Markdown, LaTeX, PNG).

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import glob
import importlib
import json
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

try:
    default_work_dir = '/content/drive/MyDrive/code2repo/kan_rec'
except ImportError:
    default_work_dir = os.path.abspath('./kan_rec')

WORK_DIR = os.environ.get('KAN_REC_DIR', default_work_dir)
DATASET = 'ml1m'  # 'amazon' | 'ml1m'
PATIENCE_ASSUMED = 4  # для запусков старого формата без поля stopped

sys.path.insert(0, WORK_DIR)
import common
importlib.reload(common)

results_dir = common.dataset_paths(WORK_DIR, DATASET)['results']
summary_dir = f'{results_dir}/summary'
os.makedirs(summary_dir, exist_ok=True)

## Загрузка результатов

In [ ]:
def model_label(saved):
    model, cfg = saved.get('model', ''), saved.get('cfg', {})
    if model.startswith('SASRec'):
        return f"SASRec-{saved.get('loss', 'bce').upper()} (h{cfg.get('hidden_units')}, d{cfg.get('dropout_rate')})"
    if model.startswith('CF-KAN'):
        if cfg.get('center', False):
            return 'CF-KAN (centered, CE)'
        return f"CF-KAN (orig, {cfg.get('loss', '?')})"
    if model.startswith('MLP-AE'):
        return model
    if model == 'BERT4Rec-kan':
        return 'KAN4Rec'
    if model == 'BERT4Rec-linear-init':
        return 'BERT4Rec (linear)'
    if model == 'BERT4Rec-linear':
        return 'BERT4Rec (linear, default init)'
    return model


def load_runs(folder):
    rows = []
    for path in sorted(glob.glob(f'{folder}/*.json')):
        filename = os.path.basename(path)
        if filename.endswith('_log.json') or 'screening' in filename:
            continue
        with open(path, encoding='utf-8') as f:
            saved = json.load(f)
        if not isinstance(saved, dict) or 'test' not in saved or 'val' not in saved:
            continue

        stem = filename[:-len('.json')]
        log = saved.get('log') or []
        last_epoch = log[-1]['epoch'] if log else None
        eval_step = log[1]['epoch'] - log[0]['epoch'] if len(log) > 1 else 1

        stopped = saved.get('stopped')
        if stopped is None and last_epoch is not None and saved.get('best_epoch') is not None:
            # запуск старого формата: причину остановки восстанавливаем по логу
            if last_epoch - saved['best_epoch'] >= eval_step * PATIENCE_ASSUMED:
                stopped = 'early_stop (по логу)'
            else:
                stopped = 'НЕ early stop (лимит времени/эпох?)'

        rows.append(dict(
            label=model_label(saved), seed=saved.get('seed', 0), stem=stem,
            params=saved.get('n_params'), sec_per_epoch=saved.get('sec_per_epoch'),
            best_epoch=saved.get('best_epoch'), last_epoch=last_epoch, stopped=stopped, gpu=saved.get('gpu'),
            has_ranks=os.path.exists(f'{folder}/ranks/{stem}_test.npy'),
            **{f'test_{k}': v for k, v in saved['test'].items()},
            **{f'val_{k}': v for k, v in saved['val'].items()},
        ))
    return pd.DataFrame(rows)


runs = load_runs(results_dir)

# если отдельного контроля с init-схемой нет, контролем считается прежний linear
if 'BERT4Rec (linear)' not in set(runs.label) and 'BERT4Rec (linear, default init)' in set(runs.label):
    runs['label'] = runs.label.replace({'BERT4Rec (linear, default init)': 'BERT4Rec (linear)'})
    print('Контроль BERT4Rec (linear) = прежний linear (init схемы репозитория не меняет веса)')

duplicated = runs[runs.duplicated(['label', 'seed'], keep=False)]
if len(duplicated):
    print('Повторы (конфигурация, seed), беру первый по имени файла:', list(duplicated.stem))
runs = runs.drop_duplicates(['label', 'seed']).reset_index(drop=True)
print(runs[['label', 'seed', 'stem', 'has_ranks']].to_string(index=False))

## Диагностика запусков
Предупреждения: запуск не закончился early stopping (мог недообучиться), лучшая эпоха совпала с последней (тоже признак
недообучения), нет рангов (запуск не попадёт в бутстреп). Время эпохи сравнимо только в пределах одной модели GPU.

In [ ]:
diagnostics = runs[['label', 'seed', 'best_epoch', 'last_epoch', 'stopped', 'sec_per_epoch', 'gpu', 'has_ranks']].copy()
diagnostics['warning'] = ''

no_early_stop = diagnostics.stopped.fillna('').str.contains('НЕ early|time_limit|max_epochs')
best_is_last = (diagnostics.best_epoch == diagnostics.last_epoch) & diagnostics.best_epoch.notna()
diagnostics.loc[no_early_stop, 'warning'] += 'не early stop; '
diagnostics.loc[best_is_last, 'warning'] += 'лучшая = последняя; '
diagnostics.loc[~diagnostics.has_ranks, 'warning'] += 'нет рангов; '

display(diagnostics.sort_values(['label', 'seed']).reset_index(drop=True))
print('запусков с предупреждениями:', int((diagnostics.warning != '').sum()), 'из', len(diagnostics))

## Итоговая таблица (test)
Сортировка по среднему test NDCG@10. Для семейства SASRec отмечена конфигурация с лучшим val NDCG@10:
с KAN-моделями сравнивайте её, а не лучшую по test.

In [ ]:
METRICS = ['Recall@10', 'NDCG@10', 'Recall@20', 'NDCG@20']


def mean_std(values):
    if len(values) > 1:
        return f'{values.mean():.4f} ± {values.std(ddof=1):.4f}'
    return f'{values.mean():.4f}'


by_model = runs.groupby('label')
table = pd.DataFrame({
    'n_seeds': by_model.size(),
    'n с рангами': by_model.has_ranks.sum(),
    **{f'test {m}': by_model[f'test_{m}'].apply(mean_std) for m in METRICS},
    'val NDCG@10': by_model['val_NDCG@10'].apply(mean_std),
    'params, M': (by_model.params.mean() / 1e6).round(1),
    'sec/epoch': by_model.sec_per_epoch.mean().round(0),
    'best epoch': by_model.best_epoch.mean().round(0),
})
table = table.loc[by_model['test_NDCG@10'].mean().sort_values(ascending=False).index]

table['выбор по val'] = ''
sasrec_labels = [label for label in by_model.groups if label.startswith('SASRec')]
if sasrec_labels:
    val_ndcg = by_model['val_NDCG@10'].mean()
    best_sasrec = max(sasrec_labels, key=lambda label: val_ndcg[label])
    table.loc[best_sasrec, 'выбор по val'] = '✓ (SASRec)'
    print('SASRec, лучшая конфигурация по val NDCG@10:', best_sasrec)

table.index.name = 'модель'
display(table)

table.to_csv(f'{summary_dir}/table.csv')
with open(f'{summary_dir}/table.tex', 'w', encoding='utf-8') as f:
    f.write(table.to_latex())
try:
    with open(f'{summary_dir}/table.md', 'w', encoding='utf-8') as f:
        f.write(table.to_markdown())
except ImportError as error:
    print('markdown-таблица пропущена:', error)

## KAN против MLP: бутстреп по пользователям
Разность `KAN − MLP` (положительная — KAN лучше). Для каждой метрики:
- значения по seed и разность по seed (среднее ± std, число seed в пользу KAN) — отражает разброс между запусками;
- 95% ДИ разности по бутстрепу пользователей (B = 1000) для каждого seed с рангами и для разности, усреднённой по ним.
  Он учитывает разброс выборки пользователей, но не разброс между seed.

Если ДИ содержит 0, а разности по seed разных знаков, различие неотличимо от шума.
Пары: CF-KAN против MLP-AE (H=256 — то же скрытое пространство, H=1792 — то же число параметров, но не та же ёмкость)
и KAN4Rec против BERT4Rec с `nn.Linear`.

In [ ]:
N_BOOTSTRAP = 1000
PAIRS = [
    ('CF-KAN (centered, CE)', 'MLP-AE (H=256)'),
    ('CF-KAN (centered, CE)', 'MLP-AE (H=1792)'),
    ('KAN4Rec', 'BERT4Rec (linear)'),
    ('KAN4Rec', 'BERT4Rec (linear, default init)'),
]


def bootstrap_ci(values, n_bootstrap=N_BOOTSTRAP, seed=0):
    rng = np.random.RandomState(seed)
    n = len(values)
    means = np.array([values[rng.randint(0, n, n)].mean() for _ in range(n_bootstrap)])
    return values.mean(), np.percentile(means, 2.5), np.percentile(means, 97.5)


def load_ranks(stem):
    return np.load(f'{results_dir}/ranks/{stem}_test.npy')


def paired_diff(stem_a, stem_b, metric):
    return common.per_user_metric(load_ranks(stem_a), metric) - common.per_user_metric(load_ranks(stem_b), metric)


def compare(label_a, label_b):
    a = runs[runs.label == label_a].set_index('seed')
    b = runs[runs.label == label_b].set_index('seed')
    shared = sorted(a.index.intersection(b.index))
    with_ranks = [s for s in shared if a.loc[s, 'has_ranks'] and b.loc[s, 'has_ranks']]

    print(f'\n=== {label_a} − {label_b}')
    print(f'    общие seed: {shared}; seed с рангами у обеих моделей: {with_ranks}')
    if not shared:
        print('    нет общих seed')
        return

    for metric in ('Recall@10', 'NDCG@10'):
        score_a, score_b = a.loc[shared, f'test_{metric}'], b.loc[shared, f'test_{metric}']
        diff = score_a - score_b
        line = f'  {metric}: {score_a.mean():.4f} против {score_b.mean():.4f}; разность по seed {diff.mean():+.4f}'
        if len(shared) > 1:
            line += f' ± {diff.std(ddof=1):.4f}'
        line += f' ({(diff / score_b).mean() * 100:+.1f}%); KAN лучше в {int((diff > 0).sum())} из {len(shared)} seed'
        print(line)

        if not with_ranks:
            print('      ранги отсутствуют, ДИ не посчитать (перезапустите эти seed)')
            continue
        for seed in with_ranks:
            mean, low, high = bootstrap_ci(paired_diff(a.loc[seed, 'stem'], b.loc[seed, 'stem'], metric))
            print(f'      seed {seed}: {mean:+.4f}  95% ДИ [{low:+.4f}, {high:+.4f}]')
        if len(with_ranks) > 1:
            averaged = np.mean([paired_diff(a.loc[s, 'stem'], b.loc[s, 'stem'], metric) for s in with_ranks], axis=0)
            mean, low, high = bootstrap_ci(averaged)
            print(f'      усреднено по {len(with_ranks)} seed: {mean:+.4f}  95% ДИ [{low:+.4f}, {high:+.4f}]')

    time_a, time_b = a.loc[shared, 'sec_per_epoch'].mean(), b.loc[shared, 'sec_per_epoch'].mean()
    params_a, params_b = a.loc[shared, 'params'].mean() / 1e6, b.loc[shared, 'params'].mean() / 1e6
    print(f'  время эпохи: {time_a:.0f} с против {time_b:.0f} с (×{time_a / time_b:.2f}); '
          f'параметры: {params_a:.1f} млн против {params_b:.1f} млн')


available = set(runs.label)
for label_a, label_b in PAIRS:
    if label_a in available and label_b in available:
        compare(label_a, label_b)

## Недетерминизм
В `results/_stale` лежат запуски, перенесённые ноутбуками 02–04 из-за несовпадения конфигурации. Старый `sasrec_ce_h128_d05_seed0` на деле
имел ту же конфигурацию и seed, что `d03`, поэтому их разность — шум одного и того же запуска (недетерминизм GPU и порядка операций).
Это нижняя граница шума: различия между моделями меньше неё различить нельзя.

In [ ]:
stale_dir = f'{results_dir}/_stale'
stale = load_runs(stale_dir) if os.path.isdir(stale_dir) else pd.DataFrame()

if stale.empty:
    print('results/_stale пуст или не создан, оценки недетерминизма нет')

for _, old in stale.iterrows():
    twins = runs[(runs.label == old.label) & (runs.seed == old.seed)]
    if twins.empty:
        print(f'{old.stem}: в results нет запуска с той же конфигурацией и seed, лежит в _stale как архив')
        continue
    current = twins.iloc[0]
    print(f'{old.stem} (в _stale) и {current.stem}: одна конфигурация ({old.label}), seed {old.seed}')
    for metric in ('Recall@10', 'NDCG@10'):
        before, after = old[f'test_{metric}'], current[f'test_{metric}']
        print(f'   test {metric}: {before:.4f} против {after:.4f}; |разность| = {abs(before - after):.4f}')

## Графики

In [ ]:
val_logs = {}
for _, run in runs.iterrows():
    saved = common.load_run(results_dir, run.stem)
    val_logs[(run.label, run.seed)] = saved.get('log') or []

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))

for (label, seed), log in val_logs.items():
    if seed == 0 and log:
        axes[0].plot([e['epoch'] for e in log], [e['NDCG@10'] for e in log], marker='o', ms=3, label=label)
axes[0].set_xlabel('эпоха')
axes[0].set_ylabel('val NDCG@10')
axes[0].set_title('Кривые валидации (seed 0)')
axes[0].legend(fontsize=7)

for label in ('KAN4Rec', 'BERT4Rec (linear)', 'CF-KAN (centered, CE)', 'MLP-AE (H=256)'):
    for seed in sorted(runs[runs.label == label].seed):
        log = val_logs.get((label, seed), [])
        if log:
            axes[1].plot([e['epoch'] for e in log], [e['NDCG@10'] for e in log], marker='o', ms=3,
                         linestyle='-' if 'KAN' in label else '--', label=f'{label} (seed {seed})')
axes[1].set_xlabel('эпоха')
axes[1].set_ylabel('val NDCG@10')
axes[1].set_title('KAN (сплошные) и контроль (пунктир), все seed')
axes[1].legend(fontsize=6, ncol=2)

plt.tight_layout()
plt.savefig(f'{summary_dir}/val_curves.png', dpi=130)
plt.show()

In [ ]:
mean_ndcg = by_model['test_NDCG@10'].mean().sort_values()
std_ndcg = by_model['test_NDCG@10'].std(ddof=1).fillna(0)
labels = list(mean_ndcg.index)
means = [mean_ndcg[label] for label in labels]
stds = [std_ndcg[label] for label in labels]

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(labels, means, xerr=stds, capsize=3)
ax.set_xlabel('test NDCG@10 (среднее ± std по seed, без std — один seed)')
for position, value in enumerate(means):
    ax.text(value, position, f' {value:.3f}', va='center', fontsize=8)

plt.tight_layout()
plt.savefig(f'{summary_dir}/test_ndcg10.png', dpi=130)
plt.show()
print('сохранено в', summary_dir)